In [2]:
import instructor
import re
import json
import os
import pandas as pd
from pydantic import BaseModel, Field
from qdrant_client import QdrantClient
from dotenv import load_dotenv
from typing import Literal
from collections import Counter

load_dotenv("../../.env")

qdrant_client = QdrantClient(url="http://localhost:6333")

SEED = 42
df = pd.read_parquet("../data/recipes_sample.parquet")

In [3]:
sample = df.sample(n=100, random_state=SEED)
remainder = df.drop(sample.index)

In [4]:
recipes = [
    f"{r.RecipeId}: " + re.sub(r"\n\s*\n", "\n", r.text)
    for r in sample.itertuples(index=False)
]

In [35]:
def build_eval_gen_prompt(n_total=30, n_single=12, n_multi=10, n_unanswerable=8):
  return f"""
You are helping build an evaluation dataset for a recipe recommendation RAG system.
The system retrieves recipes using semantic search over each recipe's full text (name, description, ingredients, instructions), then answers the user's question grounded only in the retrieved recipes.

You will be given a list of recipes: each recipe starts with its numeric id followed by a colon; recipes are separated by blank lines.

Corpus scale — the defining constraint:
- The recipes you see are a random sample from a corpus of ~5,000 similar home-cooking recipes. Assume any generic dish type (cookies, pasta, a chicken dinner, banana bread) appears dozens of times in the full corpus.
- Build each question around a distinctive COMBINATION (unusual ingredient + technique, specific occasion + dietary constraint) so that only a handful of recipes in the FULL corpus could plausibly satisfy it. Broad category questions ("a good dessert", "a chicken dinner") are not acceptable.

Generate EXACTLY {n_total} distinct questions a home cook might ask this assistant, grounded in the provided recipes. Never repeat a question, and never produce two questions whose single ground-truth recipe is the same. Speak as a real user talking about recipes — never use the words "chunk", "context" or "document".

Vary how many recipes answer each question:
- {n_single} items: answerable by a single recipe. Phrase these the way a user who WANTS that kind of dish would search — by ingredients, cuisine, dietary constraint, technique, or occasion. NEVER name the recipe's title in the question. Include enough distinguishing attributes that this recipe is the single clear best match among the provided recipes AND would plausibly stay the best match in the full corpus; if several recipes would match equally, either add a distinguishing detail or list all of them in recipe_ids.
- {n_multi} items: require combining or comparing multiple recipes — 2 or 3, never more (e.g. comparing two weeknight dinners that both use mushrooms and cream) — list ALL relevant recipe_ids.
- {n_unanswerable} items: plausible cooking questions that no recipe in the full ~5,000-recipe home-cooking corpus is likely to answer — an ingredient, cuisine, dish type or technique clearly outside this corpus's range, not merely absent from the sample you see. Return an empty recipe_ids list and, in answer_example, state that nothing available fits.

Realism rules (apply to ALL questions):
- Write as someone who does NOT know which recipes exist. Never reference "your recipes", "your list", or a specific recipe name/title. No "in the X recipe" framing.
- Vary phrasing and length across the whole set. Aim for roughly this mix: ~40% "keyword" (terse search phrases like "vegan cookies no eggs"), ~40% "natural" (a real full-sentence question, e.g. "What can I make for a quick vegetarian dinner using mushrooms?"), ~20% "detailed" (a longer multi-attribute request). Do NOT make every question a keyword phrase.
- Set query_style to honestly describe how each question is phrased: "keyword", "natural", or "detailed". The label must match the actual wording.

Vocabulary realism — the most important rule:
- Real users have never read these recipes, so they cannot echo their wording. Do NOT copy distinctive phrases or rare terms verbatim from a recipe's text. Rephrase in everyday language: "granulated sugar" becomes "sugar", "saute" becomes "fry", a flowery dish description becomes how a shopper would casually describe it.
- Describe the NEED (occasion, craving, dietary constraint, what's in the fridge) rather than enumerating a recipe's ingredient list. Mention at most 2-3 ingredients per question, named the way people talk at the grocery store.
- Never stack 4 or more exact terms lifted from a single recipe: that makes retrieval artificially easy. One or two specific words (a cuisine, one distinctive ingredient) are fine when a real user would plausibly type them.
- After paraphrasing, re-check the ground truth: the target recipe(s) must still be identifiable as the best match(es) given only the question's wording. If paraphrasing makes the question ambiguous between several recipes, list all matching recipe_ids instead.

Rules:
- Ground truth must be correct given ONLY the provided recipes. Never invent recipes or ingredients.
- Base questions and answers on the recipe TEXT only. Do NOT ask about calories, nutrition or exact cooking time — those are covered by a separate, programmatically generated bucket.
- Prefer questions with a clear, checkable ground truth.
"""


SYSTEM_PROMPT = build_eval_gen_prompt()

In [30]:
USER_PROMPT = f"Here is the list of recipes:\n\n" + "\n\n".join(recipes)

In [31]:
print(USER_PROMPT)

Here is the list of recipes:

167564.0: Dry Spice Mixture
Make and share this Dry Spice Mixture recipe from Food.com.
Ingredients: dried cayenne peppers paprika oregano thyme white pepper garlic powder
Instructions: Make it in bulk and store in a sealable glass jar. Use with caution and respect.

500430.0: Double Bran Muffins
Make and share this Double Bran Muffins recipe from Food.com.
Ingredients: brown sugar baking powder nonfat milk molasses honey canola oil walnuts
Instructions: Preheat oven to 350 degrees Fahrenheit. Place the brans and brown sugar in a medium bowl and mix well, pressing out any lumps of sugar with the back of a spoon. Add the baking powder and mix well. Combine the wet ingredients (milk though oil) And stir well, then add wet ingredients to dry mixture and mix well. Fold in optional nuts and/or fruit. Divide the mixture among 12 muffin cups which have been coated with cooking spray. Bake for about 15 minutes, until a wooden toothpick inserted in the center of a 

In [5]:
class EvalItem(BaseModel):
  reasoning: str = Field(description="Reasoning why the question can be answered with the referenced recipes.")
  question: str = Field(description="Suggested question")
  query_style: Literal["keyword", "natural", "detailed"] = Field(description="How the question is phrased: terse keywords, a natural full-sentence question, or a detailed multi-attribute request.")
  recipe_ids: list[int] = Field(description="RecipeId(s) of the recipe(s) that answer the question (ground truth for retrieval).")
  answer_example: str = Field(description="Suggested answer grounded in the context")

In [6]:
client = instructor.from_provider(
  "openai/gpt-5.4",
  mode=instructor.Mode.RESPONSES_TOOLS,
)

In [7]:
response, raw_response = client.create_with_completion(
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT},
    ],
    reasoning={"effort": "medium"},
    response_model=list[EvalItem],
)

NameError: name 'SYSTEM_PROMPT' is not defined

In [37]:
print(response)

[EvalItem(reasoning='This uniquely matches the cookie recipe that uses anise seeds, brandy, and a cinnamon-sugar finish. No other cookie here combines those holiday-style flavors.', question='anise cookies with cinnamon sugar and a little brandy', query_style='keyword', recipe_ids=[25697], answer_example='A good fit is the anise-scented sugar cookies made with eggs and a splash of brandy, then dusted with cinnamon sugar before baking.'), EvalItem(reasoning='Among the soups, this is the only one built around both barley and lentils, cooked down and strained smooth with tomato and optional milk.', question='I want a smooth soup with barley and lentils, and a little tomato in it. Is there something like that?', query_style='natural', recipe_ids=[285743], answer_example='Yes — there’s a blended barley-and-lentil soup with onion, butter, tomato puree, and optional milk stirred in at the end.'), EvalItem(reasoning='This is the only meatless sandwich built from pecan-based meatballs with cott

In [ ]:
def _norm(q: str) -> str:
    return " ".join(q.lower().split())


# Dedupe: drop repeated questions and repeated single-recipe ground truths.
# (Multi-recipe items may legitimately reuse an id, so only dedupe single-recipe ones.)
seen_questions: set[str] = set()
seen_single_ids: set[int] = set()
eval_items = []
dropped = []
for item in response:
    qkey = _norm(item.question)
    single_id = item.recipe_ids[0] if len(item.recipe_ids) == 1 else None
    if qkey in seen_questions or (single_id is not None and single_id in seen_single_ids):
        dropped.append(item)
        continue
    seen_questions.add(qkey)
    if single_id is not None:
        seen_single_ids.add(single_id)
    eval_items.append(item)

print(f"kept {len(eval_items)}, dropped {len(dropped)} duplicate(s)")

# Fail loudly if the run is malformed rather than silently saving a bad set.
bucket = Counter(
    "unanswerable" if len(i.recipe_ids) == 0 else "single" if len(i.recipe_ids) == 1 else "multi"
    for i in eval_items
)
assert bucket["single"] and bucket["multi"] and bucket["unanswerable"], f"empty bucket: {dict(bucket)}"
assert len({_norm(i.question) for i in eval_items}) == len(eval_items), "duplicate questions remain"

kept 30, dropped 0 duplicate(s)


In [40]:
os.makedirs("../data", exist_ok=True)
with open("../data/eval_dataset_large_raw.json", "w") as f:
    json.dump([item.model_dump() for item in eval_items], f, indent=2, ensure_ascii=False)

print(f"Saved {len(eval_items)} items to ../data/eval_dataset_large_raw.json")

Saved 30 items to ../data/eval_dataset_large_raw.json


### Stage 2 — relevance sweep over the remainder

Judge all 30 questions against the other ~4,883 recipes (batches of 50, ~98 calls,
`gpt-5.4-mini`) so gold labels become complete over the full corpus — this is what lets
evals run against the real collection instead of a sample copy.

Prompt layout: static prefix (rules + questions) first, recipe batch last, so the
repeated part is a cacheable prefix. Judgment is per-recipe ("alone answers it"),
with an explicit strictness clause — tested on batch 0, where the lenient version
produced 2 false positives (verified against the recipe texts).

In [10]:
import tiktoken

# Restart-safe: stage 2 reads from the checkpoint, not from `response`.
with open("../data/eval_dataset_large_raw.json") as f:
    eval_items_raw = json.load(f)

questions_block = "\n".join(f"{i}: {item['question']}" for i, item in enumerate(eval_items_raw))

SWEEP_PROMPT_PREFIX = f"""
You are a relevance annotator building ground-truth labels for a recipe recommendation RAG system.

## System being tested
The RAG system retrieves recipes using semantic search over each recipe's full text (name, description, ingredients, instructions), then answers the user's question grounded only in the retrieved recipes.

## Your input
1. A list of user questions, one per line, as "question_id: question".
2. A batch of recipes. Each recipe starts with its numeric id followed by a colon; recipes are separated by blank lines.

## Your task
For each question, identify which recipes in this batch could be used to answer it. Apply a strict, per-recipe test:
- Include a recipe id ONLY if that recipe ALONE is sufficient to answer the question on its own.
- Judge each recipe independently, never in combination with others.
- The asker has not read these recipes: a recipe qualifies if it genuinely satisfies the need expressed by the question, even when the wording differs.
- For questions that ask for multiple dishes or a comparison, include each recipe that would be one of the dishes in the answer, judged on its own merits.
- Be strict: "partially fits", "related", or "close enough" does NOT qualify. The recipe must actually be what the question asks for (a request for dips is not satisfied by another kind of appetizer). If your reasoning needs a hedge like "only partially" or "not exactly", exclude the recipe.
- If no recipe in this batch independently answers a question, return an empty list for it. Most questions will have no match in most batches - that is the expected outcome, not a failure.
- Return an entry for EVERY question id, in order.

## Questions
{questions_block}

## Recipes
"""

print(f"static prefix tokens: {len(tiktoken.get_encoding('o200k_base').encode(SWEEP_PROMPT_PREFIX))} (OpenAI cache threshold: 1024)")

static prefix tokens: 894 (OpenAI cache threshold: 1024)


In [8]:
def serialize_recipes(rows: pd.DataFrame) -> str:
    return "\n\n".join(
        f"{r.RecipeId}: " + re.sub(r"\n\s*\n", "\n", r.text)
        for r in rows.itertuples(index=False)
    )


class QuestionMatches(BaseModel):
    reasoning: str = Field(description="One short sentence: why these recipes (or none) answer the question.")
    question_id: int = Field(description="ID of the question.")
    matching_recipe_ids: list[int] = Field(description="IDs of recipes in this batch that each ALONE answer the question.")


class SweepResult(BaseModel):
    matches: list[QuestionMatches]


sweep_client = instructor.from_provider(
    "openai/gpt-5.4-mini",
    mode=instructor.Mode.RESPONSES_TOOLS,
)


def sweep_batch(batch_df: pd.DataFrame):
    result, raw = sweep_client.create_with_completion(
        messages=[{"role": "system", "content": SWEEP_PROMPT_PREFIX + serialize_recipes(batch_df)}],
        reasoning={"effort": "none"},
        response_model=SweepResult,
    )

    # Guards: a hallucinated id that slips into gold labels is undetectable later.
    batch_ids = {int(x) for x in batch_df.RecipeId}
    assert len(result.matches) == len(eval_items_raw), f"got {len(result.matches)} entries, expected {len(eval_items_raw)}"
    assert all(0 <= m.question_id < len(eval_items_raw) for m in result.matches), "question_id out of range"
    hallucinated = [i for m in result.matches for i in m.matching_recipe_ids if i not in batch_ids]
    assert not hallucinated, f"recipe ids not in batch: {hallucinated}"

    return result, raw

In [ ]:
# Single-batch test. Validated on batch 0: the lenient rule (no strictness clause)
# produced 2 false positives confirmed against the recipe texts; with it, matches align
# with the base rate: ~1-3 gold recipes / 4,883 -> ~0.6 expected hits per batch across
# all 30 questions. The full sweep should add roughly 30-100 ids total, not hundreds.
batch0 = remainder.iloc[:50]
result, raw = sweep_batch(batch0)

print(raw.usage, "\n")
hits = [m for m in result.matches if m.matching_recipe_ids]
print(f"matches in batch 0: {len(hits)} (expected ~0-2)")
for m in hits:
    print(f"Q{m.question_id}: {eval_items_raw[m.question_id]['question']}")
    print(f"  -> {m.matching_recipe_ids} | {m.reasoning}")

In [11]:
BATCH_SIZE = 50
SWEEP_CHECKPOINT = "../data/sweep_results.jsonl"

# Resume-safe: one jsonl line per judged batch. If the loop dies (network error,
# assert in sweep_batch), re-run this cell and it continues from the first missing
# batch. If you REGENERATE the questions, delete the jsonl first: it would be stale.
done = 0
if os.path.exists(SWEEP_CHECKPOINT):
    with open(SWEEP_CHECKPOINT) as f:
        done = sum(1 for _ in f)

n_batches = (len(remainder) + BATCH_SIZE - 1) // BATCH_SIZE  # ceil division
print(f"{n_batches} batches of {BATCH_SIZE}, already done: {done}")

with open(SWEEP_CHECKPOINT, "a") as f:
    for b in range(done, n_batches):
        batch = remainder.iloc[b * BATCH_SIZE : (b + 1) * BATCH_SIZE]
        result, raw = sweep_batch(batch)
        row = {
            "batch": b,
            "matches": {m.question_id: m.matching_recipe_ids for m in result.matches if m.matching_recipe_ids},
        }
        f.write(json.dumps(row) + "\n")
        f.flush()  # line hits disk now, not at close: a crash can't lose finished batches
        n_new = sum(len(v) for v in row["matches"].values())
        print(f"batch {b + 1}/{n_batches}: +{n_new} ids")

98 batches of 50, already done: 0
batch 1/98: +0 ids
batch 2/98: +1 ids
batch 3/98: +2 ids
batch 4/98: +1 ids
batch 5/98: +4 ids
batch 6/98: +3 ids
batch 7/98: +1 ids
batch 8/98: +6 ids
batch 9/98: +7 ids
batch 10/98: +1 ids
batch 11/98: +2 ids
batch 12/98: +1 ids
batch 13/98: +0 ids
batch 14/98: +3 ids
batch 15/98: +1 ids
batch 16/98: +0 ids
batch 17/98: +1 ids
batch 18/98: +1 ids
batch 19/98: +4 ids
batch 20/98: +2 ids
batch 21/98: +0 ids
batch 22/98: +1 ids
batch 23/98: +3 ids
batch 24/98: +1 ids
batch 25/98: +6 ids
batch 26/98: +3 ids
batch 27/98: +2 ids
batch 28/98: +2 ids
batch 29/98: +3 ids
batch 30/98: +3 ids
batch 31/98: +3 ids
batch 32/98: +0 ids
batch 33/98: +2 ids
batch 34/98: +1 ids
batch 35/98: +6 ids
batch 36/98: +1 ids
batch 37/98: +3 ids
batch 38/98: +6 ids
batch 39/98: +0 ids
batch 40/98: +2 ids
batch 41/98: +7 ids
batch 42/98: +0 ids
batch 43/98: +2 ids
batch 44/98: +3 ids
batch 45/98: +0 ids
batch 46/98: +0 ids
batch 47/98: +2 ids
batch 48/98: +1 ids
batch 49/98: +2

In [12]:
# Merge: sweep results + stage-1 ids, then report what changed.
question_ids = {i: set(item["recipe_ids"]) for i, item in enumerate(eval_items_raw)}

with open(SWEEP_CHECKPOINT) as f:
    for line in f:
        row = json.loads(line)
        for qid, ids in row["matches"].items():  # json keys are strings
            question_ids[int(qid)].update(ids)


def bucket_of(ids):
    return "unanswerable" if len(ids) == 0 else "single" if len(ids) == 1 else "multi"


total_added = 0
for i, item in enumerate(eval_items_raw):
    old = set(item["recipe_ids"])
    new = sorted(question_ids[i] - old)
    total_added += len(new)
    if new or bucket_of(old) != bucket_of(question_ids[i]):
        print(f"Q{i} [{bucket_of(old)} -> {bucket_of(question_ids[i])}] +{len(new)}: {item['question'][:70]}")

print(f"\ntotal ids added by the sweep: {total_added} (expected ~30-100)")

merged = [
    {**item, "recipe_ids_full": sorted(question_ids[i])}
    for i, item in enumerate(eval_items_raw)
]
with open("../data/eval_dataset_large_merged.json", "w") as f:
    json.dump(merged, f, indent=2, ensure_ascii=False)

print(f"saved {len(merged)} items to ../data/eval_dataset_large_merged.json")

Q1 [single -> multi] +1: I want a smooth soup with barley and lentils, and a little tomato in i
Q3 [single -> multi] +1: I’m after a baked shrimp pasta dish that’s extra cheesy and has a bit 
Q4 [single -> multi] +1: Asian pork dumplings with greens and a punchy dipping sauce
Q5 [single -> multi] +2: I need a baking project for kids where the cookies look like little wi
Q6 [single -> multi] +1: grilled tofu peanut sauce vegetarian
Q9 [single -> multi] +2: I’d love something like a Japanese rolled omelet with shrimp and scall
Q10 [single -> multi] +2: green tomato fries with spicy ketchup
Q11 [single -> multi] +5: I need a very simple make-ahead dessert that’s mostly soft cheese with
Q12 [multi -> multi] +5: Which baked shrimp casseroles do you have, and how are they different?
Q13 [multi -> multi] +15: banana smoothie or milkshake ideas
Q14 [multi -> multi] +18: What Greek-style chicken options are there if I want bright lemony fla
Q15 [multi -> multi] +14: peanut sauce dinner recipes


### Stage 2b — cull broad questions, verify the survivors' additions

The sweep added 258 ids — far above the healthy band. Diagnosis (run on 2026-07-09):
the stage-1 **multi bucket ignored the corpus-scale rule** (category questions like
"creamy noodle dishes" → 16–54 matches), while the single bucket held (1–7 ids) and
4 of 8 "unanswerable" questions were correctly killed by the sweep. On top of that, a
spot-check found residual judge leniency (a shrimp *pizza* labeled as a baked casserole).

Fix, cheapest-first: **cull** questions with >10 gold ids (broad = no signal: recall@5
mechanically capped, hit@5 saturated), then **verify** the remaining ~30 sweep additions
one-by-one with the strong model (funnel: cheap model scans 4,883, strong model audits 30).
The strong verifier rejected 26/30. Final: 21 questions (11 single / 3 multi / 7 unanswerable)
+ 12 constraint questions.

In [ ]:
# Funnel order matters: cull FIRST (drops most of the 258 additions with the broad
# questions), then verify only the additions on the keepers (~30 pairs, strong model).
merged = json.load(open("../data/eval_dataset_large_merged.json"))
texts = df.set_index("RecipeId")["text"]

MAX_GOLD = 10
kept, dropped_broad = [], []
for i, it in enumerate(merged):
    (kept if len(it["recipe_ids_full"]) <= MAX_GOLD else dropped_broad).append((i, it))
print(f"kept {len(kept)}, dropped {len(dropped_broad)} broad questions (>{MAX_GOLD} gold ids)")
for i, it in dropped_broad:
    print(f"  Q{i} ({len(it['recipe_ids_full'])} ids): {it['question'][:65]}")

pairs = [(i, it, rid) for i, it in kept for rid in it["recipe_ids_full"] if rid not in it["recipe_ids"]]
print(f"\n{len(pairs)} sweep additions to verify")


class Verdict(BaseModel):
    reasoning: str = Field(description="One short sentence.")
    is_gold: bool = Field(description="True only if the recipe is genuinely what the question asks for.")


VERIFY_PROMPT = """You are auditing ground-truth labels for a recipe-search eval dataset.
A cheap annotator claimed the recipe below answers the user question. Verify that claim.
Be strict: the recipe must actually be what the question asks for - "related", "partially fits" or "could work" is NOT gold. If your reasoning needs a hedge, answer false.

Question: {question}

Recipe:
{recipe}"""

verdicts = {}
for n, (i, it, rid) in enumerate(pairs, 1):
    recipe = f"{rid}: " + re.sub(r"\n\s*\n", "\n", texts.loc[rid])
    v, _ = client.create_with_completion(  # stage-1 strong-model client
        messages=[{"role": "system", "content": VERIFY_PROMPT.format(question=it["question"], recipe=recipe)}],
        reasoning={"effort": "low"},
        response_model=Verdict,
    )
    verdicts[(i, rid)] = v.is_gold
    print(f"{n:2d}/{len(pairs)} {'OK' if v.is_gold else 'NO'} Q{i} {rid}: {v.reasoning[:100]}")

final_items = []
for i, it in kept:
    ids = [rid for rid in it["recipe_ids_full"] if rid in it["recipe_ids"] or verdicts[(i, rid)]]
    final_items.append({
        "question": it["question"],
        "query_style": it["query_style"],
        "recipe_ids": ids,
        "bucket": "unanswerable" if not ids else "single" if len(ids) == 1 else "multi",
        "answer_example": it["answer_example"],
    })

print(f"\nverifier rejected {sum(1 for v in verdicts.values() if not v)}/{len(pairs)} additions")

In [ ]:
# Manual override, documented: on Q12 the verifier rejected 330839 ("Artichoke Shrimp
# Bake") with comparison semantics ("doesn't compare multiple casseroles") while
# conceding it IS a baked shrimp casserole - text checked by hand, it belongs in gold.
q12 = next(i for i in final_items if i["question"].startswith("Which baked shrimp casseroles"))
if 330839 not in q12["recipe_ids"]:
    q12["recipe_ids"].append(330839)
    q12["bucket"] = "multi"

with open("../data/eval_dataset_large_final.json", "w") as f:
    json.dump(final_items, f, indent=2, ensure_ascii=False)

print(Counter(i["bucket"] for i in final_items))
print(f"saved {len(final_items)} items to ../data/eval_dataset_large_final.json")

In [ ]:
# Constraint bucket: same 12 questions as notebook 06, ground truth recomputed over
# the FULL corpus (zero LLM cost - computed from the numeric columns).
CONSTRAINT_ITEMS = [
    {"question": "light dessert under 200 calories", "query_style": "keyword",
     "constraints": [{"field": "Calories", "op": "lt", "value": 200}]},
    {"question": "What can I make for breakfast that has less than 300 calories?", "query_style": "natural",
     "constraints": [{"field": "Calories", "op": "lt", "value": 300}]},
    {"question": "dinner main dish below 500 kcal", "query_style": "keyword",
     "constraints": [{"field": "Calories", "op": "lt", "value": 500}]},
    {"question": "I only have 20 minutes, what can I cook that's ready in under 20 minutes?", "query_style": "natural",
     "constraints": [{"field": "total_time_minutes", "op": "lt", "value": 20}]},
    {"question": "something I can make in half an hour or less", "query_style": "natural",
     "constraints": [{"field": "total_time_minutes", "op": "lte", "value": 30}]},
    {"question": "high protein meal with at least 30 grams of protein", "query_style": "natural",
     "constraints": [{"field": "ProteinContent", "op": "gte", "value": 30}]},
    {"question": "protein-packed dish, 20g protein or more", "query_style": "keyword",
     "constraints": [{"field": "ProteinContent", "op": "gte", "value": 20}]},
    {"question": "hearty filling meal with more than 500 calories", "query_style": "natural",
     "constraints": [{"field": "Calories", "op": "gt", "value": 500}]},
    {"question": "slow cooked recipe that takes over 2 hours", "query_style": "natural",
     "constraints": [{"field": "total_time_minutes", "op": "gt", "value": 120}]},
    {"question": "low calorie snack, max 150 kcal", "query_style": "keyword",
     "constraints": [{"field": "Calories", "op": "lte", "value": 150}]},
    {"question": "quick lunch ready in under 30 minutes and below 400 calories", "query_style": "detailed",
     "constraints": [{"field": "total_time_minutes", "op": "lt", "value": 30},
                     {"field": "Calories", "op": "lt", "value": 400}]},
    {"question": "I want a high protein dinner, more than 20 grams of protein but under 600 calories", "query_style": "detailed",
     "constraints": [{"field": "ProteinContent", "op": "gt", "value": 20},
                     {"field": "Calories", "op": "lt", "value": 600}]},
]

OPS = {"lt": lambda c, v: c < v, "lte": lambda c, v: c <= v,
       "gt": lambda c, v: c > v, "gte": lambda c, v: c >= v}


def matching_ids(constraints):
    mask = pd.Series(True, index=df.index)
    for c in constraints:
        col = df[c["field"]]
        mask &= col.notna() & OPS[c["op"]](col, c["value"])
    return [int(x) for x in df.loc[mask, "RecipeId"]]


for item in CONSTRAINT_ITEMS:
    item["matching_ids"] = matching_ids(item["constraints"])
    assert item["matching_ids"], f"no recipe satisfies: {item['question']!r}"
    print(f"{len(item['matching_ids']):4d} recipes satisfy: {item['question']!r}")

In [ ]:
# Write the COMPLETE assembled eval dataset to ONE tool-neutral file - the single
# source of truth. Pushing it to a tracking tool (Langfuse) is a separate, thin step
# (evals/upload_dataset.py), so the dataset is never coupled to a vendor again.
#
# Sourced WITHOUT re-running the expensive LLM verify: final_items is the persisted
# verified result, constraint items (+ matching_ids) come from the cell above (cheap,
# no LLM). Falls back to disk so this runs on a cold kernel after cell 0 + the
# constraint cell.
import json

try:
    final_items
except NameError:
    final_items = json.load(open("../data/eval_dataset_large_final.json"))
try:
    texts
except NameError:
    texts = df.set_index("RecipeId")["text"]

records = []
for it in final_items:
    ids = [int(r) for r in it["recipe_ids"]]
    records.append({
        "question": it["question"],
        "bucket": it["bucket"],
        "query_style": it["query_style"],
        "answer_example": it["answer_example"],
        "reference_context_ids": ids,
        "reference_description": [str(texts.loc[r]) for r in ids],
        "constraints": [],
        "constraint_matching_ids": [],
    })
for it in CONSTRAINT_ITEMS:
    records.append({
        "question": it["question"],
        "bucket": "constraint",
        "query_style": it["query_style"],
        "answer_example": None,
        "reference_context_ids": [],
        "reference_description": [],
        "constraints": it["constraints"],
        "constraint_matching_ids": [int(r) for r in it["matching_ids"]],
    })

out_path = "../data/eval_dataset_large.jsonl"
with open(out_path, "w") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

n_constraint = sum(1 for r in records if r["bucket"] == "constraint")
print(f"wrote {len(records)} examples ({n_constraint} constraint) to {out_path}")